### Imports and R Environment Setup


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings

# Filter out the specific rpy2 environment variable warnings
warnings.filterwarnings("ignore", category=UserWarning, message='.*Environment variable ".*" redefined by R.*')

In [4]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Core project imports
# from graphical_sampling.sampling import KMeansSampler
from graphical_sampling.population import Population
from package_sampling.utils import inclusion_probabilities

# Your Python-based index modules from the index folder
from graphical_sampling.index import Density
from graphical_sampling.index import Moran 
from graphical_sampling.index import Voronoi 
from graphical_sampling.index import LocalBalance 

### Optimized Sampling Functions

In [5]:
# B
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter

# Define the converter context for rpy2
combined_converter = ro.default_converter + numpy2ri.converter + pandas2ri.converter

def run_sampling_design(method, coords, probs, n, num_samples):
    N = len(coords)
    
    # 1. Python Methods (Nmcs and Rand)
    if method == "Nmcs":
        # Create the population and sampler
        pop = Population(coords=coords, probs=probs)
        sampler = KMeansSampler(
            population=pop, 
            n=n, 
            n_zones=(3, 3), 
            zone_builder="sweep",
            units_order="spiral",
            zones_order="spiral"
        )
        # Return samples for simulation AND the sampler for math
        return sampler.sample(num_samples), sampler

    if method == "Rand":
        samples_idx = np.zeros((num_samples, n), dtype=int)
        for i in range(num_samples):
            samples_idx[i] = np.random.choice(N, n, replace=False) #
        return samples_idx, None

    # 2. R Methods (Lopi, Wave, Maxe, Scps)
    samples_idx = np.zeros((num_samples, n), dtype=int)
    
    with localconverter(combined_converter):
        ro.globalenv['coords_r'] = coords
        ro.globalenv['probs_r'] = probs
        
        ro.r("library(BalancedSampling)")
        ro.r("library(WaveSampling)")
        ro.r("library(sampling)")
        
        for i in range(num_samples):
            if method == "Lopi":
                samples_idx[i] = np.array(ro.r("lpm2(probs_r, coords_r)")) - 1 #
            elif method == "Scps":
                samples_idx[i] = np.array(ro.r("scps(probs_r, coords_r)")) - 1 #
            elif method == "Wave":
                mask = ro.r("wave(coords_r, probs_r)") #
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0] #
            elif method == "Maxe":
                mask = ro.r("sampling::UPmaxentropy(probs_r)") #
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0] #

    return samples_idx, None

### Metrics and Spread Calculation

In [6]:
# C
import numpy as np

def calculate_all_scores(coords, probs, sample_idx, n, N, scorers, y_val, method):
    """
    Calculates spatial and statistical scores using Python modules.
    
    Args:
        coords: Population coordinates.
        probs: Inclusion probabilities.
        sample_idx: 1D array of selected unit indices.
        n: Sample size.
        N: Population size.
        scorers: Dictionary of initialized Density, Moran, Voronoi, and LocalBalance objects.
        y_val: The variable of interest for estimation.
        method: The name of the sampling method.
    """
    # 1. Prepare index for Python (ensure 2D shape for .score method)
    s_idx_2d = sample_idx.reshape(1, -1)
    
    # 2. Estimator Calculation
    if method == "Rand":
        # SRS Estimator for random sampling
        est_val = N * np.mean(y_val[sample_idx])
    else:
        # HT Estimator: Sum(y_i / pi_i)
        est_val = np.sum(y_val[sample_idx] / probs[sample_idx])
    
    # 3. Calculate Spatial Scores using Python modules
    # Accessing pre-initialized scorers from the dictionary to avoid re-calculation
    dens_score = scorers['D'].score(s_idx_2d)[0]
    moran_score = scorers['M'].score(s_idx_2d)[0]
    voronoi_score = scorers['V'].score(s_idx_2d)[0]
    lb_score = scorers['L'].score(s_idx_2d)[0]

    # Return Order: Density, Voronoi, Moran, Local Balance, Estimator
    return (
        dens_score, 
        voronoi_score, 
        moran_score, 
        lb_score, 
        est_val
    )

### The Main Execution Loop

In [7]:
# # Data Selection
# import os
# import pandas as pd
# import numpy as np
# import graphical_sampling as gs

# # Choose your population
# pop_name = ["rand", 'clust', 'grid', 'meuse'] # Options: "meuse", "swiss", "rand"
# n_size = [4, 8, 16]

# # 1. Start looking in the current directory
# data_folder = "populations"

# # 2. Keep stepping one folder up (../) until it finds it
# for _ in range(3):
#     if os.path.exists(data_folder):
#         break
#     data_folder = "../" + data_folder  # <-- FIXED TYPO

# # 3. Safely define results folder relative to the data folder's location
# base_dir = os.path.dirname(data_folder) if os.path.dirname(data_folder) else "."
# results_folder = os.path.join(base_dir, "simulations", "results")

# # 4. Read the file
# # df = pd.read_csv(f"{data_folder}/{pop_name}.csv")
# # coords = df[["x", "y"]].values.astype(float)

# # if pop_name == 'meuse':
# #     y_values = df["cadmium"].values
# #     pik = inclusion_probabilities(df["copper"].values, n_size)
# # elif pop_name == 'swiss':
# #     y_values = df["AREA_A"].values
# #     pik = inclusion_probabilities(df["AREA"].values, n_size)
# # else:
# #     y_values = df['z.90'].values
# #     pik = inclusion_probabilities(df["prob"].values, n_size)

# # # Create the population object once
# # pop_wrapped = Population(coords=coords, probs=pik)
# # true_sum_y = np.sum(y_values)

# # print(f"Data Loaded: {pop_name} | N={len(df)} | Target Mean={np.mean(y_values):.4f}")

In [8]:
# import pandas as pd
# import os

# # 1. Define your specific populations
# pop_names = ["meuse", "swiss", "rand"]
# dfs = {}

# # 2. Loop through the list of names
# for pop in pop_names:
#     path = os.path.join(data_folder, f'{pop}.csv')
#     dfs[pop] = pd.read_csv(path)
    
#     corr_matrix = dfs[pop].corr(numeric_only=True)
#     # print(f'\n{pop} Correlation Matrix =\n', corr_matrix.round(2))

In [9]:
# --- High-Efficiency Main Configuration and Loop ---
import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

# (Assuming your custom imports like Population, Density, Moran, Voronoi, 
# LocalBalance, run_sampling_design, inclusion_probabilities are up here)

# 1. Configuration Switches & Setup
INCLUDE_NMCS_IN_SIM = False  # Use theoretical math for Nmcs to save time
FIND_PIK_EXPLOSION = False
sample_cnt = 5000            # Increased sample count improves simulation stability

# pop_names = ['rand', 'grid', 'clust', 'meuse']
pop_names = ['swiss']
n_sizes = [50, 100, 150]
# Silence rpy2 warnings
warnings.filterwarnings("ignore", category=UserWarning, module="rpy2")

# --- Smart Folder Setup ---
data_folder = "populations"
for _ in range(3):
    if os.path.exists(data_folder):
        break
    data_folder = "../" + data_folder

base_dir = os.path.dirname(data_folder) if os.path.dirname(data_folder) else "."
results_folder = os.path.join(base_dir, "simulations", "results")
os.makedirs(results_folder, exist_ok=True) # Ensure results folder exists

# (Make sure pop_names and n_sizes are defined in your notebook!)
# Example: pop_names = ["meuse", "swiss", "rand"]
# Example: n_sizes = [10]
inclusion = "equal"
for name in pop_names:
    for n_size in n_sizes:
        n = n_size
        # 2. Load and Prep Data
        file_path = os.path.join(data_folder, f"{name}.csv")
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue
            
        df = pd.read_csv(file_path)
        coords = df[["x", "y"]].values.astype(float)
        
        # Assign target variable and inclusion probabilities
        if name == 'meuse':
            y_values = df["cadmium"].values
            pik = inclusion_probabilities(df["copper"].values, n_size)
        elif name == 'swiss':
            y_values = df["AREA_A"].values
            y_values = y_values.clip(5,100)
            p_values = df["AREA"].values.clip(5, 100)
            pik = inclusion_probabilities(p_values, n_size)
            # print("y_value",y_values)
            # print("p_value",p_values)
        else:
            y_values = df['z.90'].values
            pik = inclusion_probabilities(df["prob"].values, n_size)
        
        N = len(df)
        true_sum_y = np.sum(y_values)
        if inclusion == "equal":
            pik = inclusion_probabilities(np.ones(N), n_size)
            rho = 0
        else: 
            rho = np.corrcoef(y_values, pik)[0, 1]

        # # 1. Force probabilities slightly away from absolute 0 and 1
        # pik = np.clip(pik, 1e-7, 1 - 1e-7)
        
        # # 2. Force the array to sum exactly to 'n' to fix floating-point drift
        # pik = pik * (n / np.sum(pik))

        
        

            
        print(f"\n--- Processing {name} (N={N}, n={n}, True Total={true_sum_y:.3f}, Rho={rho:.3f}) ---")

        # --- DEBUG BLOCK: Theoretical SRS Benchmark ---
        S2_y = np.var(y_values, ddof=1) 
        theoretical_srs_var = (N**2) * (1 - n/N) * (S2_y / n)
        print(f"DEBUG: Theoretical SRS Total Variance: {theoretical_srs_var:.2f}")

        # 3. Setup Metrics (One-time initialization)
        pop_wrapped = Population(coords=coords, probs=pik)
        scorer_D = Density(population=pop_wrapped, n=n)
        scorer_M = Moran(population=pop_wrapped)
        scorer_V = Voronoi(population=pop_wrapped)
        scorer_L = LocalBalance(population=pop_wrapped)

        # 4. Sampling and Scoring Loop
        all_data = []
        methods = ["Lopi", "Scps", "Rand", "Maxe"] 
        if INCLUDE_NMCS_IN_SIM:
            methods.insert(0, "Nmcs")

        for m in methods:
            print(f"Running simulation for {m}...")
            samples, _ = run_sampling_design(m, coords, pik, n, sample_cnt)
            
            # Vectorized scoring (High Efficiency)
            d_scores = scorer_D.score(samples)
            m_scores = scorer_M.score(samples)
            v_scores = scorer_V.score(samples)
            l_scores = scorer_L.score(samples)
            
            # Calculate Estimators
            if m == "Rand":
                est_scores = N * np.mean(y_values[samples], axis=1)
            else:
                est_scores = np.sum(y_values[samples] / pik[samples], axis=1)

            # Store results for EVERY iteration
            for i in range(sample_cnt):
                all_data.append([N, n, sample_cnt, rho, m, d_scores[i], v_scores[i], m_scores[i], l_scores[i], est_scores[i]])
                
                # --- TRACKING RARE EXTREME ESTIMATES ---
                if FIND_PIK_EXPLOSION:
                    error_margin = abs(est_scores[i] - true_sum_y) / true_sum_y
                    if error_margin > 0.5:
                        print(f"\n[ALERT] Method: {m} | Iteration: {i}")
                        print(f"Estimate: {est_scores[i]:.2f} | True Total: {true_sum_y:.2f} | Error: {error_margin:.2%}")
                        
                        bad_sample_idx = samples[i]
                        bad_y = y_values[bad_sample_idx]
                        bad_pik = pik[bad_sample_idx]
                        
                        contributions = bad_y / bad_pik
                        culprit_local_idx = np.argmax(contributions)
                        culprit_pop_idx = bad_sample_idx[culprit_local_idx]
                        
                        print(f"Sample Indices: {bad_sample_idx}")
                        print(f"Exploded Unit Index: {culprit_pop_idx} | y: {bad_y[culprit_local_idx]} | pik: {bad_pik[culprit_local_idx]}")

        # 5. Data Processing & Aggregation
        res_df = pd.DataFrame(all_data, columns=["N", "n", "ite" ,"rho", "Method", "D", "V", "M", "L", "HT"])
        
        # Aggregate simulation results (ignores N, n, rho because we group by Method only on target cols)
        summary = res_df.groupby("Method").agg({
            "D": ["mean", "std"], 
            "V": ["mean", "std"], 
            "M": ["mean", "std"], 
            "L": ["mean", "std"], 
            "HT": ["mean", "var"]
        })

        # CRITICAL: Force column order before renaming
        summary = summary[["D", "V", "M", "L", "HT"]]
        summary.columns = ["Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls", "HTm", "HTv"]

        # 6. Inject Theoretical Nmcs row (If available)
        theoretical_results = None
        if theoretical_results:
            t = theoretical_results
            summary.loc["Nmcs"] = [
                t['Exp_Density'], t['SD_Density'], 
                t['Exp_Voronoi'], t['SD_Voronoi'], 
                t['Exp_Moran'], t['SD_Moran'], 
                t['Exp_Local'], t['SD_Local'], 
                true_sum_y, t['HT_Variance']
            ]

        # 7. Final Metrics: RB and Efficiency (Eff)
        summary["RB"] = (summary["HTm"] - true_sum_y) / true_sum_y
        if "Rand" in summary.index:
            rand_var = summary.loc["Rand", "HTv"]
            summary["Eff"] = rand_var / summary["HTv"].replace(0, np.nan)
        else:
            summary["Eff"] = np.nan
            
        summary['rho'] = rho
        summary['ite'] = sample_cnt
        summary['n'] = n
        summary['N'] = N
        # Final Table Output formatting
        final_cols = ['N', 'n', 'ite','rho', "HTm", "HTv" ,"Eff", "RB", "Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls"]
        summary = summary.reindex(columns=final_cols)
        print(summary.round(3))

        # 8. Save Data
        # A. Save individual summary
        summary_file = os.path.join(results_folder, f"summary_{name}_n={n}_corr={rho:.1f}.csv")
        summary.to_csv(summary_file)
        
        # B. Save extended raw iterations (Append Mode)
        extended_file = os.path.join(results_folder, "all_iterations_extended.csv")
        res_df.to_csv(extended_file, mode='a', index=False, header=not os.path.exists(extended_file))

print("\nAll simulations completed and saved!")

R callback write-console: Loading required package: Matrix
  



--- Processing swiss (N=959, n=50, True Total=10562.630, Rho=0.000) ---
DEBUG: Theoretical SRS Total Variance: 5163349.82
Running simulation for Lopi...


Running simulation for Scps...
Running simulation for Rand...
Running simulation for Maxe...
          N   n   ite  rho        HTm          HTv    Eff     RB     Dm  \
Method                                                                    
Lopi    959  50  5000    0  10518.759  4851786.922  1.078 -0.004  0.029   
Maxe    959  50  5000    0  10608.201  5170824.727  1.011  0.004  0.005   
Rand    959  50  5000    0  10541.388  5229955.864  1.000 -0.002 -0.003   
Scps    959  50  5000    0  10537.619  4991318.320  1.048 -0.002  0.030   

           Ds     Vm     Vs     Mm     Ms     Lm     Ls  
Method                                                   
Lopi    0.064  0.126  0.025 -0.246  0.034  0.088  0.010  
Maxe    0.183  0.421  0.114 -0.007  0.043  0.164  0.026  
Rand    0.184  0.424  0.114 -0.007  0.044  0.165  0.027  
Scps    0.063  0.120  0.023 -0.284  0.031  0.086  0.010  

--- Processing swiss (N=959, n=100, True Total=10562.630, Rho=0.000) ---
DEBUG: Theoretical SRS Total Varia

### Plots

In [10]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Filter the data: Remove the 'Rand' (SRS) method
plot_df_eff = summary.drop('Rand')
plot_df_spread = summary.drop('Nmcs')


# Set up the figure with 2 subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Generate some colors dynamically based on how many methods are left
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(plot_df_spread)))

# ==========================================
# PART 1: Efficiency Plot
# ==========================================
axes[0].bar(plot_df_eff.index, plot_df_eff['Eff'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Sampling Efficiency Relative to SRS', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Efficiency (Var_SRS / Var_Design)', fontsize=12)
axes[0].set_xlabel('Sampling Method', fontsize=12)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of the bars
for i, v in enumerate(plot_df_eff['Eff']):
    axes[0].text(i, v + (plot_df_eff['Eff'].max() * 0.02), f'{v:.2f}', 
                 ha='center', va='bottom', fontweight='bold')

# ==========================================
# PART 2: Spread Indices Plot with Error Bars
# ==========================================
# Define the metrics and their corresponding mean/std columns in your DataFrame
metrics = ['Density (D)', 'Voronoi (V)', 'Moran (M)', 'Local Bal (L)']
means_cols = ['Dm', 'Vm', 'Mm', 'Lm']
stds_cols = ['Ds', 'Vs', 'Ms', 'Ls']

x = np.arange(len(metrics))  # Label locations
num_methods = len(plot_df_eff)
total_width = 0.8          # Total width available for a group of bars
width = total_width / num_methods  # Width of a single bar

# Plot bars for each method dynamically
plot_idx = 0
for i, method in enumerate(plot_df_spread.index):

    # Extract means and standard deviations directly from the dataframe
    method_means = plot_df_spread.loc[method, means_cols].values
    method_stds = plot_df_spread.loc[method, stds_cols].values
    
    # Calculate position so bars group nicely around the center of the tick
    pos = x - (total_width / 2) + (i * width) + (width / 2)
    
    axes[1].bar(pos, method_means, width, yerr=method_stds, label=method, 
                color=colors[i], edgecolor='black', capsize=5, alpha=0.85)

# Formatting the Spread Plot
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics, fontsize=11)
axes[1].set_title('Spread Indices (Mean ± Std Dev)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Index Value', fontsize=12)
axes[1].legend(title="Method", title_fontsize='12', fontsize='11')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=1) # Reference line at 0

# Final layout adjustments
plt.tight_layout()
plt.show()

KeyError: "['Nmcs'] not found in axis"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the data (excluding 'Rand' / SRS)
data = {
    'Method': ['Lopi', 'Nmcs', 'Scps'],
    'Eff': [5.020, 3.720, 5.155],
    'Dm': [0.088, 0.047, 0.090],
    'Ds': [0.189, 0.064, 0.180],
    'Vm': [0.125, 0.072, 0.112],
    'Vs': [0.078, 0.054, 0.072],
    'Mm': [-0.180, -0.216, -0.202],
    'Ms': [0.076, 0.092, 0.067],
    'Lm': [0.519, 0.537, 0.507],
    'Ls': [0.177, 0.228, 0.167]
}

df = pd.DataFrame(data)

# Set up the figure with 2 subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Define colors for consistent styling
colors = ['#4C72B0', '#55A868', '#C44E52']

# ==========================================
# PART 1: Efficiency Plot
# ==========================================
axes[0].bar(df['Method'], df['Eff'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Sampling Efficiency Relative to SRS', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Efficiency (Higher is better)', fontsize=12)
axes[0].set_xlabel('Method', fontsize=12)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of the bars
for i, v in enumerate(df['Eff']):
    axes[0].text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')


# ==========================================
# PART 2: Spread Indices Plot with Error Bars
# ==========================================
# Define the metrics and their corresponding mean/std columns
metrics = ['Density (D)', 'Voronoi (V)', 'Moran (M)', 'Local Balance (L)']
means_cols = ['Dm', 'Vm', 'Mm', 'Lm']
stds_cols = ['Ds', 'Vs', 'Ms', 'Ls']

x = np.arange(len(metrics))  # Label locations
width = 0.25  # Width of the bars

# Plot bars for each method
for i, method in enumerate(df['Method']):
    # Extract means and standard deviations for the current method
    method_means = df.loc[df['Method'] == method, means_cols].values[0]
    method_stds = df.loc[df['Method'] == method, stds_cols].values[0]
    
    # Calculate position (shift left, center, shift right)
    pos = x - width + (i * width)
    
    axes[1].bar(pos, method_means, width, yerr=method_stds, label=method, 
                color=colors[i], edgecolor='black', capsize=5, alpha=0.85)

# Formatting the Spread Plot
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics, fontsize=11)
axes[1].set_title('Spread Indices (Mean ± Std Dev)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Index Value', fontsize=12)
axes[1].legend(title="Method", title_fontsize='12', fontsize='11')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=1) # Reference line at 0

# Final layout adjustments
plt.tight_layout()
plt.show()

# Store

In [ ]:
# This cell acts as a barrier
raise SystemExit("Stopping Run All: Archived cells below.")

In [ ]:

import pandas as pd
import os

folder = "/config/ws/graphical-sampling/populations"
populations = ['clust', 'grid', 'rand', 'meuse', 'swiss']

dfs = {}
for pop in populations:
    path = os.path.join(folder, f'{pop}.csv')
    dfs[pop] = pd.read_csv(path)
    # Changed 'df' to 'dfs[pop]' to match the loaded data
    corr_matrix = dfs[pop].corr(numeric_only=True)
    
    # Updated print to show the name and the matrix clearly
    print('\n', pop, 'Correlation Matrix = \n', corr_matrix.round(2))

In [ ]:
import pandas as pd
import os

folder = "/config/ws/graphical-sampling/populationssss"

# --- 1. Process Meuse ---
meuse_path = os.path.join(folder, "meuse_full.csv")
if os.path.exists(meuse_path):
    df_meuse = pd.read_csv(meuse_path)
    # Keep only the requested columns
    df_meuse = df_meuse[['x', 'y', 'copper', 'cadmium', 'lead', 'zinc']]
    # Save back to folder
    df_meuse.to_csv(os.path.join(folder, "meuse.csv"), index=False)
    print("✅ Meuse modified and saved.")

# --- 2. Process Swiss ---
swiss_path = os.path.join(folder, "swiss_full.csv")
if os.path.exists(swiss_path):
    df_swiss = pd.read_csv(swiss_path)
    # Keep specific columns
    df_swiss = df_swiss[['COORD_X', 'COORD_Y', 'AREA', 'AREA_A', 'AREA_B']]
    # Rename coordinates to x and y
    df_swiss = df_swiss.rename(columns={'COORD_X': 'x', 'COORD_Y': 'y'})
    # Save back to folder
    df_swiss.to_csv(os.path.join(folder, "swiss.csv"), index=False)
    print("✅ Swiss modified and saved.")

In [ ]:
import numpy as np
import pandas as pd
import os

# 1. Define your file list based on the image provided
# Note: I am assuming the folder name is 'data_samples' based on previous context.
# If they are in the current directory, change folder to "."
folder = "/config/ws/graphical-sampling/populations"

pop_files = {
    'clust_eq':   'clust_eq_N=100.csv',
    'clust_uneq': 'clust_uneq_N=100.csv',
    'grid_eq':    'grid_eq_N=100.csv',
    'grid_uneq':  'grid_uneq_N=100.csv',
    'random_eq':  'random_eq_N=100.csv',
    'random_uneq':'random_uneq_N=100.csv',
}

# Helper function to generate correlated variables
def generate_correlated_variable(v, correlation, seed=None):
    """Generates a new variable correlated with vector v at a specific r."""
    if seed: np.random.seed(seed)
    # 1. Create random noise
    noise = np.random.normal(0, 1, len(v))
    
    # 2. Standardize target v to remove mean/scale effects for calculation
    v_norm = (v - np.mean(v)) / np.std(v)
    
    # 3. Residualize noise (make it orthogonal to v)
    # This step ensures exact mathematical control over correlation
    noise_resid = noise - (np.dot(noise, v_norm) / np.dot(v_norm, v_norm)) * v_norm
    noise_norm = noise_resid / np.std(noise_resid)
    
    # 4. Combine to get desired correlation
    # New = r * Old + sqrt(1-r^2) * Noise
    new_var = correlation * v_norm + np.sqrt(1 - correlation**2) * noise_norm
    
    # 5. Rescale back to original range (optional, but good for probability-like vars)
    # Here we just shift it to be positive to act as a "size" variable
    new_var = new_var - np.min(new_var) + 0.1 
    return new_var

data_store = {}

print("--- Loading & Processing Data ---")
for key, fname in pop_files.items():
    path = os.path.join(folder, fname)
    
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        # Basic extractions
        coords = df[['x', 'y']].values
        probs = df['prob'].values
        N = len(df)
        
        # --- LOGIC: Handle "uneq" vs "eq" files ---
        # We look for "uneq" in the key name
        if "uneq" in key:
            print(f"Processing {key}: Generating correlated auxiliaries...")
            
            # Generate the 3 auxiliary variables
            # These act as 'Target Y' variables with different correlations to inclusion probs
            y_70 = generate_correlated_variable(probs, 0.70, seed=42)
            y_80 = generate_correlated_variable(probs, 0.80, seed=43)
            y_90 = generate_correlated_variable(probs, 0.90, seed=44)
            
            # Store them so we can loop over them later
            targets_dict = {
                'y_70': y_70,
                'y_80': y_80,
                'y_90': y_90
            }
        else:
            # For Equal Probability (EP), Prob is constant. 
            # Correlation with a constant is undefined/zero. 
            # We just create one synthetic target to test spatial balance.
            print(f"Processing {key}: Standard EP file.")
            synthetic_y = (df['x'] + df['y']) + np.random.normal(0, 1, N)
            targets_dict = {'y_synthetic': synthetic_y}

        # Save to data_store
        data_store[key] = {
            'coords': coords,
            'probs': probs,
            'targets': targets_dict, # Now holds multiple Ys
            'N': N
        }
    else:
        print(f"⚠️ Warning: File {fname} not found in {folder}. Skipping.")
        
print("✅ Data Loaded Successfully.")

In [ ]:
# Save the modified data to new CSVs
output_folder = "modified_populations"
os.makedirs(output_folder, exist_ok=True)

print("\n--- Saving All Generated Variables ---")
for key, data in data_store.items():
    # 1. Start with the base spatial and probability data
    df_output = pd.DataFrame(data['coords'], columns=['x', 'y'])
    df_output['prob'] = data['probs']
    
    # 2. Add every target stored in the dictionary
    for target_name, target_values in data['targets'].items():
        # Map the internal name (y_70) to your requested format (z.70)
        # We split by underscore and join with a dot
        formatted_name = target_name.replace("y_", "z.") 
        
        df_output[formatted_name] = target_values
        
    # 3. Save the file
    save_path = os.path.join(output_folder, f"{key}_modified.csv")
    df_output.to_csv(save_path, index=False)
    
    # 4. Feedback on what was saved
    saved_cols = [c for c in df_output.columns if 'z.' in c or 'synthetic' in c]
    print(f"Saved {key}: Included variables {saved_cols}")

print(f"\n✅ All files saved successfully in: {output_folder}")

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(sp)
data(meuse)        # Loads the dataframe
data(meuse.grid)   # Loads the prediction grid
data(meuse.riv)    # Loads the river boundaries

# Convert to simple features (modern format)
library(sf)
meuse_sf <- st_as_sf(meuse, coords = c("x", "y"), crs = 28992)
meuse_sf

In [ ]:
%%R
# Drop geometry
meuse_df <- st_drop_geometry(meuse_sf)

# Keep only numeric columns
numeric_vars <- meuse_df[sapply(meuse_df, is.numeric)]

# Compute correlation matrix
cor_matrix <- cor(numeric_vars, use = "complete.obs")

cor_matrix


In [ ]:
%%R
# Extract coordinates
coords <- st_coordinates(meuse_sf)

# Drop geometry and bind coordinates
meuse_df <- cbind(
  st_drop_geometry(meuse_sf),
  x = coords[,1],
  y = coords[,2]
)

# Convert to plain data frame (optional but safe)
meuse_df <- as.data.frame(meuse_df)

head(meuse_df)



In [ ]:
%%R
# Extract coordinates and drop geometry column
coords <- st_coordinates(meuse_sf)
meuse_df <- cbind(st_drop_geometry(meuse_sf), x = coords[,1], y = coords[,2])

# Save it as a CSV (in your working directory)
write.csv(meuse_df, "meuse_with_coords.csv", row.names = FALSE)


In [ ]:
import os
import pandas as pd

# Load your final CSV from R
meuse_df = pd.read_csv("meuse_with_coords.csv")

# Create folder
output_folder = "modified_populations"
os.makedirs(output_folder, exist_ok=True)

# Save it
meuse_df.to_csv(f"{output_folder}/meuse_modified.csv", index=False)

print("Saved successfully.")
